# Spike Synchronization Analysis

Analysis of spike synchronization metrics between cerebellar populations (Mossy Fibers and Deep Cerebellar Nuclei) across different lesion conditions, including:
- **Phase-Locking Value (PLV)** of DCN spikes relative to the MF gamma oscillation
- **Cross-correlation analysis** of MF→DCN and PC→DCN pathways (spike-based and rate-based)
- **Gamma-band cross-correlation** from bandpass-filtered population firing rates
- **Amplitude Envelope Correlation (AEC)** in the gamma band
- **Transfer Entropy** from MF to DCN population rates
- **Statistical comparisons** across conditions with FDR correction and Cohen's d effect sizes

**Authors:** Alice Geminiani ([alice.geminiani@unipv.it](mailto:alice.geminiani@unipv.it)) and GitHub Copilot with Claude Opus 4.6

In [ ]:
import numpy as np
from scipy.stats import mannwhitneyu
import os

import matplotlib.pyplot as plt
import matplotlib.cm as cm

plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'figure.dpi': 150,
})

from utils.file_utils import load_pickled_dict
from utils.analysis_utils import (
    extract_spike_lists,
    population_rate,
    spike_train_binary,
    compute_internal_synchrony,
    compute_cross_population_synchrony,
    compute_synchronization_index,
    compute_plv,
    compute_rayleigh_pvalue,
    compute_spike_cross_correlation,
    extract_spike_phases,
    compute_sta,
    plot_clean_overlay,
    plot_cross_spectral_density_mean_std,
    get_significance_marker,
    transfer_entropy_with_shuffle,
    compute_aec_standard,
    darken_color
)

############################################
# -------- USER INPUT ----------------------
############################################

conditions = ['COSIM', 'COSIM_MOStoDCN', 'COSIM_PKJtoDCN', 'COSIM_INHtoPKJ', 'COSIM_MLItoMLI']
sim_runs = range(10)

norm_dir = "COSIM_NEST_LESIONrate_ampl_norm"
transient = 2.5        # seconds of initial transient to exclude
dt = 0.001            # 1 ms binning
fs = 1 / dt            # sampling frequency (1000 Hz)
gamma_band = (25, 60)  # Hz
sync_bin_size = 0.010  # 20 ms bin for synchronization measures

control = conditions[0]
test_conditions = conditions[1:]
n_cond = len(conditions)

hemispheres = ['Left', 'Right']

############################################
# -------- Load spikes from pickle ---------
# Each cell type is loaded per hemisphere
############################################

spikes_all = {cond: {} for cond in conditions}

for condition in conditions:
    for run in sim_runs:
        data_dir = os.path.join('publication_data', 'NESTlesions', norm_dir, 'spikes data', f'nsd{run}')
        file_name = f'{condition}_Spikes.pkl'
        file_path = os.path.join(data_dir, file_name)

        if not os.path.exists(file_path):
            print(f"WARNING: {file_path} not found, skipping.")
            continue

        spikes_data = load_pickled_dict(file_path)

        run_data = {}
        for hemi in hemispheres:
            run_data[f'DCN_{hemi}'] = extract_spike_lists(spikes_data, 'dcn_cell_glut_large', hemisphere=hemi, t_min=transient)
            run_data[f'MF_{hemi}']  = extract_spike_lists(spikes_data, 'mossy_fibers', hemisphere=hemi, t_min=transient)
            run_data[f'PC_{hemi}']  = extract_spike_lists(spikes_data, 'purkinje_cell', hemisphere=hemi, t_min=transient)

        all_spike_times = np.concatenate(
            [s for hemi in hemispheres
             for key in [f'DCN_{hemi}', f'MF_{hemi}', f'PC_{hemi}']
             for s in run_data[key]]
        )
        duration = np.ceil(all_spike_times.max())
        run_data['duration'] = duration

        spikes_all[condition][run] = run_data

        for hemi in hemispheres:
            print(f"[{condition}] nsd{run} {hemi}: "
                  f"DCN={len(run_data[f'DCN_{hemi}'])} neurons "
                  f"({sum(len(s) for s in run_data[f'DCN_{hemi}'])} spikes), "
                  f"MF={len(run_data[f'MF_{hemi}'])} neurons "
                  f"({sum(len(s) for s in run_data[f'MF_{hemi}'])} spikes), "
                  f"PC={len(run_data[f'PC_{hemi}'])} neurons "
                  f"({sum(len(s) for s in run_data[f'PC_{hemi}'])} spikes), "
                  f"duration={duration:.0f} s")

In [ ]:
############################################
# -------- Compute all metrics -------------
# Compute per hemisphere, then average
############################################

metrics = {cond: {} for cond in conditions}

scalar_keys = ['PC_internal_sync', 'DCN_internal_sync',
               'MF_PC_cross_sync', 'MF_DCN_cross_sync',
               'PC_sync_index', 'DCN_sync_index',
               'PLV_PC_to_PC', 'PLV_DCN_to_DCN',
               'PLV_MF_to_PC', 'PLV_MF_to_DCN']

for condition in conditions:
    for run in sorted(spikes_all[condition].keys()):
        data = spikes_all[condition][run]
        duration = data['duration']

        hemi_results = {}
        lags_ref = None

        for hemi in hemispheres:
            DCN_spikes = data[f'DCN_{hemi}']
            MF_spikes  = data[f'MF_{hemi}']
            PC_spikes  = data[f'PC_{hemi}']

            cn_flat = np.concatenate(DCN_spikes)
            mf_flat = np.concatenate(MF_spikes)
            pc_flat = np.concatenate(PC_spikes)

            DCN_rate, time_vec = population_rate(DCN_spikes, duration, dt)
            MF_rate, _         = population_rate(MF_spikes, duration, dt)
            PC_rate, _         = population_rate(PC_spikes, duration, dt)

            DCN_internal_sync = compute_internal_synchrony(DCN_spikes, duration, bin_size=sync_bin_size)
            PC_internal_sync  = compute_internal_synchrony(PC_spikes, duration, bin_size=sync_bin_size)

            # Cross-population pairwise correlation
            MF_PC_cross_sync  = compute_cross_population_synchrony(MF_spikes, PC_spikes, duration, bin_size=sync_bin_size)
            MF_DCN_cross_sync = compute_cross_population_synchrony(MF_spikes, DCN_spikes, duration, bin_size=sync_bin_size)

            t_start = transient
            t_stop  = duration
            DCN_sync_index = compute_synchronization_index(DCN_spikes, t_start, t_stop, bin_size=sync_bin_size)
            PC_sync_index  = compute_synchronization_index(PC_spikes, t_start, t_stop, bin_size=sync_bin_size)

            PLV_PC_to_PC   = compute_plv(pc_flat, PC_rate, fs, gamma_band)
            PLV_DCN_to_DCN = compute_plv(cn_flat, DCN_rate, fs, gamma_band)
            PLV_MF_to_DCN  = compute_plv(cn_flat, MF_rate, fs, gamma_band)
            PLV_MF_to_PC   = compute_plv(pc_flat, MF_rate, fs, gamma_band)

            lags, xcorr_mf = compute_spike_cross_correlation(mf_flat, cn_flat, duration)
            _,    xcorr_pc = compute_spike_cross_correlation(pc_flat, cn_flat, duration)
            if lags_ref is None:
                lags_ref = lags

            hemi_results[hemi] = {
                'PC_internal_sync':  PC_internal_sync,
                'DCN_internal_sync': DCN_internal_sync,
                'MF_PC_cross_sync':  MF_PC_cross_sync,
                'MF_DCN_cross_sync': MF_DCN_cross_sync,
                'PC_sync_index':     PC_sync_index,
                'DCN_sync_index':    DCN_sync_index,
                'PLV_PC_to_PC':      PLV_PC_to_PC,
                'PLV_DCN_to_DCN':    PLV_DCN_to_DCN,
                'PLV_MF_to_PC':      PLV_MF_to_PC,
                'PLV_MF_to_DCN':     PLV_MF_to_DCN,
                'xcorr_mf':          xcorr_mf,
                'xcorr_pc':          xcorr_pc,
            }

        # Average scalar metrics across hemispheres
        avg = {}
        for key in scalar_keys:
            avg[key] = np.mean([hemi_results[h][key] for h in hemispheres])
        # Average cross-correlation arrays across hemispheres
        avg['lags']     = lags_ref
        avg['xcorr_mf'] = np.mean([hemi_results[h]['xcorr_mf'] for h in hemispheres], axis=0)
        avg['xcorr_pc'] = np.mean([hemi_results[h]['xcorr_pc'] for h in hemispheres], axis=0)

        metrics[condition][run] = avg

        print(f"[{condition}] nsd{run}: "
              f"PC_sync={avg['PC_internal_sync']:.4f}, "
              f"DCN_sync={avg['DCN_internal_sync']:.4f}, "
              f"MF–PKJ_sync={avg['MF_PC_cross_sync']:.4f}, "
              f"MF–DCN_sync={avg['MF_DCN_cross_sync']:.4f}, "
              f"PLV_PC→PC={avg['PLV_PC_to_PC']:.4f}, "
              f"PLV_DCN→DCN={avg['PLV_DCN_to_DCN']:.4f}, "
              f"PLV_MF→PC={avg['PLV_MF_to_PC']:.4f}, "
              f"PLV_MF→DCN={avg['PLV_MF_to_DCN']:.4f} (hemisphere avg)")

print("\nAll metrics computed (hemisphere-averaged).")

In [ ]:
############################################
# -------- Statistical Comparison ----------
# All scalar metrics, each condition vs control
############################################

from NESTlesions.plot_utils import add_stat_annotation

scalar_metric_names = ['PC_internal_sync', 'DCN_internal_sync',
                       'MF_PC_cross_sync', 'MF_DCN_cross_sync',
                       'PLV_PC_to_PC', 'PLV_DCN_to_DCN',
                       'PLV_MF_to_PC', 'PLV_MF_to_DCN']
metric_labels = {
    'PC_internal_sync':  'PKJ internal synchrony\n(pairwise corr)',
    'DCN_internal_sync': 'DCN internal synchrony\n(pairwise corr)',
    'MF_PC_cross_sync':  'MF–PKJ cross synchrony\n(pairwise corr)',
    'MF_DCN_cross_sync': 'MF–DCN cross synchrony\n(pairwise corr)',
    'PLV_PC_to_PC':      'PKJ–PKJ gamma PLV',
    'PLV_DCN_to_DCN':    'DCN–DCN gamma PLV',
    'PLV_MF_to_PC':      'MF–PKJ gamma PLV',
    'PLV_MF_to_DCN':     'MF–DCN gamma PLV',
}

# Collect values per condition
cond_values = {}
for cond in conditions:
    cond_values[cond] = {m: [] for m in scalar_metric_names}
    for run in sorted(metrics[cond].keys()):
        for m in scalar_metric_names:
            cond_values[cond][m].append(metrics[cond][run][m])

# ---------- Print summary table ----------
cond_headers = f"{'Metric':<40} {control:>14}"
for tc in test_conditions:
    cond_headers += f" {tc:>18} {'p-value':>10}"
sep = "=" * len(cond_headers)

print(sep)
print(cond_headers)
print(sep)

p_values = {tc: {} for tc in test_conditions}

for m in scalar_metric_names:
    vals_ctrl = np.array(cond_values[control][m])
    line = f"{metric_labels[m].replace(chr(10), ' '):<40} "
    line += f"{np.mean(vals_ctrl):>7.4f}±{np.std(vals_ctrl):.4f}  "

    for tc in test_conditions:
        vals_test = np.array(cond_values[tc][m])
        if len(vals_ctrl) >= 2 and len(vals_test) >= 2:
            stat, p = mannwhitneyu(vals_ctrl, vals_test, alternative='two-sided')
        else:
            p = np.nan
        p_values[tc][m] = p
        sig = '*' if p < 0.05 else ''
        line += f"{np.mean(vals_test):>7.4f}±{np.std(vals_test):.4f}  p={p:.4f} {sig}  "

    print(line)

print(sep)
print("Mann-Whitney U test (two-sided). * = p < 0.05")

# ---------- Bar plot with individual points ----------
# 2x4 layout: top row = DCN measures, bottom row = PKJ measures
# Columns: internal pairwise corr | MF cross pairwise corr | internal gamma PLV | MF gamma PLV
metric_order = [
    'DCN_internal_sync', 'MF_DCN_cross_sync', 'PLV_DCN_to_DCN', 'PLV_MF_to_DCN',   # top row: DCN
    'PC_internal_sync',  'MF_PC_cross_sync',  'PLV_PC_to_PC',   'PLV_MF_to_PC',     # bottom row: PKJ
]
n_metrics = len(metric_order)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

# Use consistent lesion condition colors and labels
lesion_colors = {
    'COSIM':          'darkgray',
    'COSIM_PKJtoDCN': 'lightblue',
    'COSIM_MOStoDCN': 'darkblue',
    'COSIM_INHtoPKJ': 'darkgreen',
    'COSIM_MLItoMLI': 'lightgreen',
}
lesion_labels = {
    'COSIM':          'CONTROL',
    'COSIM_PKJtoDCN': 'PCtoCN',
    'COSIM_MOStoDCN': 'MOStoCN',
    'COSIM_INHtoPKJ': 'MLItoPC',
    'COSIM_MLItoMLI': 'MLItoMLI',
}
colors = lesion_colors
bar_width = 0.5

for ax, m in zip(axes, metric_order):
    means = [np.mean(cond_values[c][m]) for c in conditions]
    sds   = [np.std(cond_values[c][m])  for c in conditions]
    x_pos = np.arange(n_cond)

    ax.bar(x_pos, means, yerr=sds, capsize=5, width=bar_width,
           color=[colors[c] for c in conditions], alpha=0.7,
           edgecolor='black', linewidth=0.8)

    # Overlay individual run data points
    for ci, cond in enumerate(conditions):
        vals = cond_values[cond][m]
        jitter = np.random.default_rng(42).uniform(-0.1, 0.1, size=len(vals))
        ax.scatter(np.full(len(vals), ci) + jitter, vals,
                   color=darken_color(colors[cond]), s=12, zorder=5, alpha=0.8)

    # X-axis labels
    xlabels = [lesion_labels.get(c, c) for c in conditions]
    ax.set_xticks(x_pos)
    ax.set_xticklabels(xlabels, rotation=25, ha='right')
    ax.set_title(metric_labels[m])
    ax.set_ylabel('Value')

    # Annotate p-values using add_stat_annotation from plot_utils
    max_y = max(means[i] + sds[i] for i in range(n_cond))
    spacing = 0.08 * max_y
    for ti, tc in enumerate(test_conditions):
        p = p_values[tc][m]
        ci_test = conditions.index(tc)
        add_stat_annotation(ax, 0, ci_test, max_y, p, h=spacing * (ti + 1))

    # Adjust y-limit to accommodate annotations
    ax.set_ylim(top=max_y + spacing * (len(test_conditions) + 2))

comparisons_str = ', '.join([f'{control} vs {tc}' for tc in test_conditions])
fig.suptitle(f'Spike Synchronization Metrics\n{comparisons_str}',
             fontweight='bold', y=1.02)
plt.tight_layout()

import os
from NESTlesions.plot_utils import save_figure_multi_format
fig_output_dir = os.path.join('NESTlesions', norm_dir, 'spike synch analysis')
os.makedirs(fig_output_dir, exist_ok=True)
save_figure_multi_format(fig, os.path.join(fig_output_dir, 'spike_synchronization_metrics'))
print(f"Figure saved to: {fig_output_dir}/spike_synchronization_metrics.[png|eps|svg]")

plt.show()

## Phase-Locking Value (PLV) Visualization

Rose plots of DCN spike phase distributions relative to the MF gamma oscillation, a time-domain overlay of the gamma wave with spike events, and a bar plot comparing PLV across lesion conditions.

In [ ]:
############################################
# -------- PLV Visualization ---------------
# Rose plots of DCN spike phases relative
# to MF gamma, per condition
# + Rayleigh test for phase uniformity
# All computed per hemisphere then averaged
############################################

import os
from NESTlesions.plot_utils import save_figure_multi_format, add_stat_annotation

# ============================================================
# 1.  Compute spike phases & Rayleigh test for every
#     (condition, run) pair — per hemisphere, then average PLV
# ============================================================

phases_all   = {cond: [] for cond in conditions}   # list of 1-D phase arrays per run (both hemispheres concatenated)
plv_all      = {cond: [] for cond in conditions}   # scalar PLV per run (hemisphere-averaged)
rayleigh_all = {cond: [] for cond in conditions}   # Rayleigh p-value per run (on concatenated phases)

for condition in conditions:
    for run in sorted(spikes_all[condition].keys()):
        data = spikes_all[condition][run]
        hemi_plvs = []
        hemi_phases = []

        for hemi in hemispheres:
            mf_rate, _ = population_rate(data[f'MF_{hemi}'], data['duration'], dt)
            dcn_flat   = np.concatenate(data[f'DCN_{hemi}'])

            spike_ph, _ = extract_spike_phases(dcn_flat, mf_rate, fs, gamma_band)
            hemi_phases.append(spike_ph)
            hemi_plvs.append(np.abs(np.mean(np.exp(1j * spike_ph))))

        # Concatenate phases from both hemispheres for rose plots / Rayleigh
        run_phases = np.concatenate(hemi_phases)
        phases_all[condition].append(run_phases)
        # Average PLV across hemispheres
        plv_all[condition].append(np.mean(hemi_plvs))
        rayleigh_all[condition].append(compute_rayleigh_pvalue(run_phases))

print("Spike phases extracted for all conditions / runs (hemisphere-averaged).\n")

# ---- Print Rayleigh test summary table ----
print("=" * 90)
print(f"{'Condition':<18} {'Run':>4} {'PLV':>8} {'Rayleigh p':>14} {'Significant?':>14}")
print("-" * 90)
for cond in conditions:
    for ri, run in enumerate(sorted(spikes_all[cond].keys())):
        plv_val = plv_all[cond][ri]
        ray_p   = rayleigh_all[cond][ri]
        sig     = '***' if ray_p < 0.001 else ('**' if ray_p < 0.01 else ('*' if ray_p < 0.05 else 'ns'))
        print(f"{lesion_labels[cond]:<18} {run:>4} {plv_val:>8.4f} {ray_p:>14.2e} {sig:>14}")
print("=" * 90)

# Pooled Rayleigh test per condition
print("\nPooled Rayleigh test (all runs concatenated):")
print("-" * 60)
for cond in conditions:
    pooled_ph = np.concatenate(phases_all[cond])
    pooled_ray_p = compute_rayleigh_pvalue(pooled_ph)
    pooled_plv   = np.abs(np.mean(np.exp(1j * pooled_ph)))
    sig = '***' if pooled_ray_p < 0.001 else ('**' if pooled_ray_p < 0.01 else ('*' if pooled_ray_p < 0.05 else 'ns'))
    print(f"  {lesion_labels[cond]:<16}  PLV={pooled_plv:.4f}  Rayleigh p={pooled_ray_p:.2e}  {sig}")
print("-" * 60)

# ============================================================
# 2a. Rose-plot grid: RAW pooled phase histograms
#     with Rayleigh p-value in subtitle
# ============================================================

num_bins = 24
bins_rose = np.linspace(-np.pi, np.pi, num_bins + 1)

fig_rose, axes_rose = plt.subplots(1, n_cond, figsize=(5 * n_cond, 5),
                                    subplot_kw={'projection': 'polar'})
if n_cond == 1:
    axes_rose = [axes_rose]

for ci, cond in enumerate(conditions):
    ax = axes_rose[ci]
    # Pool phases across all runs for this condition
    pooled = np.concatenate(phases_all[cond])
    counts, edges = np.histogram(pooled, bins=bins_rose, density=True)
    centres = (edges[:-1] + edges[1:]) / 2
    widths  = np.diff(edges)

    col = lesion_colors[cond]
    ax.bar(centres, counts, width=widths, bottom=0.0,
           color=col, alpha=0.75, edgecolor='black', linewidth=0.8)

    # Mean resultant vector (arrow showing preferred phase & PLV)
    mean_vec = np.mean(np.exp(1j * pooled))
    mean_angle = np.angle(mean_vec)
    ax.annotate('', xy=(mean_angle, counts.max() * 0.95),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=2.5))

    # Formatting
    ax.set_theta_zero_location('E')
    ax.set_theta_direction(1)
    ax.set_xticks(np.pi / 180. * np.linspace(0, 360, 8, endpoint=False))
    ax.set_xticklabels([r'$0$', r'$\pi/4$', r'$\pi/2$', r'$3\pi/4$',
                        r'$\pm\pi$', r'$-3\pi/4$', r'$-\pi/2$', r'$-\pi/4$'],
                       fontsize=10)
    ax.set_yticklabels([])

    # PLV & Rayleigh p in title
    mean_plv = np.mean(plv_all[cond])
    pooled_ray_p = compute_rayleigh_pvalue(pooled)
    sig_marker = '***' if pooled_ray_p < 0.001 else ('**' if pooled_ray_p < 0.01 else ('*' if pooled_ray_p < 0.05 else 'ns'))
    ax.set_title(f'{lesion_labels[cond]}\nPLV = {mean_plv:.3f}\n'
                 f'Rayleigh p = {pooled_ray_p:.2e} {sig_marker}',
                 fontsize=12, pad=22, fontweight='bold')

fig_rose.suptitle('DCN Spike Phase Distribution Relative to MF Gamma\n'
                   f'(pooled across {len(sim_runs)} runs & hemispheres, {gamma_band[0]}–{gamma_band[1]} Hz)',
                   fontsize=16, fontweight='bold', y=1.10)
plt.tight_layout()

fig_output_dir = os.path.join('NESTlesions', norm_dir, 'spike synch analysis')
os.makedirs(fig_output_dir, exist_ok=True)
save_figure_multi_format(fig_rose, os.path.join(fig_output_dir, 'PLV_rose_plots'))
print(f"\nRose plots saved to: {fig_output_dir}/PLV_rose_plots.[png|eps|svg]")
plt.show()

# ============================================================
# 2b. Rose-plot grid: Z-SCORED phase histograms
# ============================================================

fig_rose_z, axes_rose_z = plt.subplots(1, n_cond, figsize=(5 * n_cond, 5),
                                        subplot_kw={'projection': 'polar'})
if n_cond == 1:
    axes_rose_z = [axes_rose_z]

for ci, cond in enumerate(conditions):
    ax_z = axes_rose_z[ci]
    pooled = np.concatenate(phases_all[cond])
    counts, edges = np.histogram(pooled, bins=bins_rose, density=True)
    centres = (edges[:-1] + edges[1:]) / 2
    widths  = np.diff(edges)

    # Z-score the bin counts
    z_counts = (counts - counts.mean()) / (counts.std() + 1e-12)

    # Shift z-scores so the minimum is at 0 for polar bar plotting
    z_shifted = z_counts - z_counts.min()

    # Color bars by z-score: red for high, blue for low
    z_norm = plt.Normalize(vmin=z_counts.min(), vmax=z_counts.max())
    z_cmap = plt.cm.RdBu_r
    bar_colors = z_cmap(z_norm(z_counts))

    ax_z.bar(centres, z_shifted, width=widths, bottom=0.0,
             color=bar_colors, alpha=0.85, edgecolor='black', linewidth=0.8)

    # Mean resultant vector arrow
    mean_vec = np.mean(np.exp(1j * pooled))
    mean_angle = np.angle(mean_vec)
    ax_z.annotate('', xy=(mean_angle, z_shifted.max() * 0.95),
                  xytext=(0, 0),
                  arrowprops=dict(arrowstyle='->', color='red', lw=2.5))

    ax_z.set_theta_zero_location('E')
    ax_z.set_theta_direction(1)
    ax_z.set_xticks(np.pi / 180. * np.linspace(0, 360, 8, endpoint=False))
    ax_z.set_xticklabels([r'$0$', r'$\pi/4$', r'$\pi/2$', r'$3\pi/4$',
                          r'$\pm\pi$', r'$-3\pi/4$', r'$-\pi/2$', r'$-\pi/4$'],
                         fontsize=10)
    ax_z.set_yticklabels([])
    ax_z.set_title(f'{lesion_labels[cond]} (Z-scored)\nZ range: [{z_counts.min():.2f}, {z_counts.max():.2f}]',
                   fontsize=11, pad=18, fontweight='bold')

fig_rose_z.suptitle('DCN Spike Phase Distribution Relative to MF Gamma (Z-scored)\n'
                     f'(pooled across {len(sim_runs)} runs & hemispheres, {gamma_band[0]}–{gamma_band[1]} Hz)',
                     fontsize=16, fontweight='bold', y=1.10)
plt.tight_layout()

save_figure_multi_format(fig_rose_z, os.path.join(fig_output_dir, 'PLV_rose_plots_Zscored'))
print(f"Z-scored rose plots saved to: {fig_output_dir}/PLV_rose_plots_Zscored.[png|eps|svg]")
plt.show()

# ============================================================
# 3.  Time-domain overlay: MF gamma wave + DCN spikes
#     Using plot_clean_overlay for CONTROL, MOStoDCN, PKJtoDCN
#     (uses Right hemisphere for visualization)
# ============================================================

# --- Plot overlays for CONTROL, MOStoDCN, and PKJtoDCN ---
overlay_conditions = ['COSIM', 'COSIM_MOStoDCN', 'COSIM_PKJtoDCN']

# 100 ms window starting 0.5 s after transient (in ms)
window_start_ms = (transient + 0.5) * 1000   # e.g. 3000 ms
window_end_ms   = window_start_ms + 100       # e.g. 3100 ms

for ov_cond in overlay_conditions:
    rep_run = sorted(spikes_all[ov_cond].keys())[0]
    rep_data = spikes_all[ov_cond][rep_run]

    # Use Right hemisphere for the time-domain overlay visualization
    mf_rate_rep, t_rep = population_rate(rep_data['MF_Right'], rep_data['duration'], dt)
    _, gamma_wave_rep = extract_spike_phases(
        np.concatenate(rep_data['DCN_Right']), mf_rate_rep, fs, gamma_band)

    # Convert time to ms
    t_rep_ms = t_rep * 1000

    # Convert per-neuron DCN spike times to ms
    dcn_spike_list_ms = [spk * 1000 for spk in rep_data['DCN_Right']]

    overlay_title = (f'{lesion_labels[ov_cond]} – MF Gamma & DCN Spikes\n'
                     f'({gamma_band[0]}–{gamma_band[1]} Hz, run {rep_run})')

    fig_ov = plot_clean_overlay(
        t_rep_ms, gamma_wave_rep, dcn_spike_list_ms,
        window_start=window_start_ms, window_end=window_end_ms,
        num_neurons=20, title=overlay_title
    )

    # Save each overlay figure
    save_name = f'PLV_time_overlay_{lesion_labels[ov_cond]}'
    save_figure_multi_format(fig_ov, os.path.join(fig_output_dir, save_name))
    print(f"Overlay saved to: {fig_output_dir}/{save_name}.[png|eps|svg]")
    plt.show()

# ============================================================
# 3b. Spike-Triggered Average: MF gamma aligned to DCN spikes
#     Per hemisphere, then averaged; mean ± STD across runs
# ============================================================

# --- Compute per-run STA for every (condition, run) pair ---
sta_window_ms = 50
sta_per_run = {cond: [] for cond in conditions}   # {cond: list of 1-D STA arrays}
sta_time_axis = None

for cond in conditions:
    for run in sorted(spikes_all[cond].keys()):
        run_data = spikes_all[cond][run]

        hemi_stas = []
        for hemi in hemispheres:
            mf_rate_run, _ = population_rate(run_data[f'MF_{hemi}'], run_data['duration'], dt)
            _, gamma_wave_run = extract_spike_phases(
                np.concatenate(run_data[f'DCN_{hemi}']), mf_rate_run, fs, gamma_band)

            dcn_flat_run = np.concatenate(run_data[f'DCN_{hemi}'])
            sta_run, t_ax_run = compute_sta(gamma_wave_run, dcn_flat_run, fs,
                                            window_ms=sta_window_ms)
            hemi_stas.append(sta_run)

            if sta_time_axis is None:
                sta_time_axis = t_ax_run

        # Average STA across hemispheres for this run
        sta_per_run[cond].append(np.mean(hemi_stas, axis=0))

    print(f"[{lesion_labels[cond]}] STA computed for {len(sta_per_run[cond])} runs (hemisphere-averaged)")

# --- Per-condition mean ± STD across runs ---
sta_mean = {}
sta_std  = {}

for cond in conditions:
    stack = np.array(sta_per_run[cond])        # (n_runs, n_time)
    sta_mean[cond] = stack.mean(axis=0)
    sta_std[cond]  = stack.std(axis=0)


# --- Combined STA comparison: all conditions on one plot ---
fig_comb, ax_comb = plt.subplots(figsize=(10, 6))
ax_comb.axvline(0, color='red', linestyle='--', linewidth=2, label='DCN Spike (t=0)')

for cond in conditions:
    col = lesion_colors[cond]
    lab = lesion_labels[cond]

    # Find peak delay for legend label
    peak_idx = np.argmax(sta_mean[cond])
    peak_delay = sta_time_axis[peak_idx]

    ax_comb.plot(sta_time_axis, sta_mean[cond], color=col, linewidth=2,
                 label=f'{lab} (peak {peak_delay:+.1f} ms)')
    ax_comb.fill_between(sta_time_axis,
                         sta_mean[cond] - sta_std[cond],
                         sta_mean[cond] + sta_std[cond],
                         color=col, alpha=0.15)

ax_comb.set_title(f'Spike-Triggered Average – All Conditions\n'
                  f'MF Gamma ({gamma_band[0]}–{gamma_band[1]} Hz), '
                  f'mean ± STD across {len(sim_runs)} runs',
                  fontsize=15, fontweight='bold')
ax_comb.set_xlabel('Time relative to DCN spike (ms)', fontsize=12)
ax_comb.set_ylabel('MF Gamma Amplitude', fontsize=12)
ax_comb.set_xlim(-sta_window_ms, sta_window_ms)
ax_comb.grid(True, linestyle='--', alpha=0.6)
ax_comb.legend(loc='upper right')
plt.tight_layout()

save_figure_multi_format(fig_comb, os.path.join(fig_output_dir, 'STA_all_conditions'))
print(f"Combined STA saved to: {fig_output_dir}/STA_all_conditions.[png|eps|svg]")
plt.show()


# ============================================================
# 4.  Bar plot of PLV (MF→DCN) per condition with stats
#     + Rayleigh significance markers on each bar
# ============================================================

fig_bar, ax_bar = plt.subplots(figsize=(8, 5.5))
x_pos = np.arange(n_cond)
bar_means = [np.mean(plv_all[c]) for c in conditions]
bar_stds  = [np.std(plv_all[c])  for c in conditions]
bar_cols  = [lesion_colors[c] for c in conditions]

ax_bar.bar(x_pos, bar_means, yerr=bar_stds, capsize=5, width=0.55,
           color=bar_cols, alpha=0.7, edgecolor='black', linewidth=0.8)

rng = np.random.default_rng(42)
for ci, cond in enumerate(conditions):
    vals = np.array(plv_all[cond])
    jitter = rng.uniform(-0.12, 0.12, size=len(vals))
    ax_bar.scatter(np.full(len(vals), ci) + jitter, vals,
                   color=darken_color(bar_cols[ci]), s=14, zorder=5, alpha=0.8)

    # Add Rayleigh significance marker below each bar label
    pooled_ph = np.concatenate(phases_all[cond])
    ray_p = compute_rayleigh_pvalue(pooled_ph)
    ray_sig = '***' if ray_p < 0.001 else ('**' if ray_p < 0.01 else ('*' if ray_p < 0.05 else 'ns'))
    ax_bar.text(ci, -0.008 * max(bar_means), f'R: {ray_sig}',
                ha='center', va='top', fontsize=9, color='darkred', fontweight='bold')

ax_bar.set_xticks(x_pos)
ax_bar.set_xticklabels([lesion_labels[c] for c in conditions], rotation=15, ha='right')
ax_bar.set_ylabel('Phase-Locking Value (PLV)')
ax_bar.set_title(f'MF→DCN Gamma PLV ({gamma_band[0]}–{gamma_band[1]} Hz)\n'
                 f'Mean ± STD across {len(sim_runs)} runs  |  R: Rayleigh test',
                 fontweight='bold')
ax_bar.grid(True, axis='y', linestyle='--', alpha=0.5)

# Statistical annotations (each lesion vs control, independent t-test)
from scipy.stats import ttest_ind as _ttest
ctrl_vals = np.array(plv_all[control])
max_bar_y = max(bar_means[i] + bar_stds[i] for i in range(n_cond))
spacing = 0.06 * max_bar_y
for ti, tc in enumerate(test_conditions):
    test_vals = np.array(plv_all[tc])
    _, p_plv = _ttest(ctrl_vals, test_vals)
    ci_test = conditions.index(tc)
    add_stat_annotation(ax_bar, 0, ci_test, max_bar_y, p_plv, h=spacing * (ti + 1))

ax_bar.set_ylim(top=max_bar_y + spacing * (len(test_conditions) + 2))

plt.tight_layout()
save_figure_multi_format(fig_bar, os.path.join(fig_output_dir, 'PLV_barplot'))
print(f"Bar plot saved to: {fig_output_dir}/PLV_barplot.[png|eps|svg]")
plt.show()

## Gamma-Band Cross-Correlation (Rate-Based)

Cross-correlation computed from **population firing rates filtered in the gamma band** (25–60 Hz), comparing MF→DCN across all lesion conditions. Both MF and DCN rates are bandpass-filtered before computing the normalized cross-correlation.

In [ ]:
############################################
# -------- Gamma-Band Cross-Correlation ----
# Cross-correlation of MF and DCN population
# rates filtered in the gamma band
# (hemisphere-averaged, all conditions)
############################################

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, correlate as scipy_correlate
from NESTlesions.plot_utils import save_figure_multi_format

# Bandpass filter helper
def _bandpass_filter(signal, fs, band, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [band[0] / nyq, band[1] / nyq], btype='band')
    return filtfilt(b, a, signal)

# Compute gamma-filtered cross-correlation per condition/run (hemisphere-averaged)
gamma_xcorr_mf_dcn = {cond: [] for cond in conditions}
gamma_xcorr_lags = None

for condition in conditions:
    for run in sorted(spikes_all[condition].keys()):
        data = spikes_all[condition][run]
        duration = data['duration']

        hemi_xcorr = []

        for hemi in hemispheres:
            mf_rate, _ = population_rate(data[f'MF_{hemi}'], duration, dt)
            dcn_rate, _ = population_rate(data[f'DCN_{hemi}'], duration, dt)

            # Bandpass filter both signals in the gamma band
            mf_gamma = _bandpass_filter(mf_rate, fs, gamma_band)
            dcn_gamma = _bandpass_filter(dcn_rate, fs, gamma_band)

            # Z-score for normalized (Pearson-like) cross-correlation
            mf_z = (mf_gamma - mf_gamma.mean()) / (mf_gamma.std() + 1e-12)
            dcn_z = (dcn_gamma - dcn_gamma.mean()) / (dcn_gamma.std() + 1e-12)

            n = len(mf_z)
            xcorr = scipy_correlate(dcn_z, mf_z, mode='full') / n
            hemi_xcorr.append(xcorr)

            if gamma_xcorr_lags is None:
                gamma_xcorr_lags = np.arange(-(n - 1), n) * dt  # in seconds

        gamma_xcorr_mf_dcn[condition].append(np.mean(hemi_xcorr, axis=0))

print("Gamma-band cross-correlations computed (hemisphere-averaged).\n")

# ---- Plot: all conditions overlapped ----
fig_gamma_xcorr, ax_gx = plt.subplots(figsize=(10, 5))

max_lag_display = 0.05  # ±50 ms
mask_gx = (gamma_xcorr_lags >= -max_lag_display) & (gamma_xcorr_lags <= max_lag_display)
lags_ms_gx = gamma_xcorr_lags[mask_gx] * 1000

for condition in conditions:
    stack_gx = np.array(gamma_xcorr_mf_dcn[condition])[:, mask_gx]
    mean_gx = stack_gx.mean(axis=0)
    sem_gx = stack_gx.std(axis=0) / np.sqrt(len(stack_gx))

    col = lesion_colors[condition]
    lab = lesion_labels[condition]
    ax_gx.plot(lags_ms_gx, mean_gx, color=col, linewidth=2, label=lab)
    ax_gx.fill_between(lags_ms_gx, mean_gx - sem_gx, mean_gx + sem_gx,
                        color=col, alpha=0.15)

ax_gx.axvline(0, color='grey', linestyle='--', linewidth=1)
ax_gx.set_xlabel('Lag (ms)', fontsize=12)
ax_gx.set_ylabel('Cross-correlation (r)', fontsize=12)
ax_gx.set_title(f'Gamma-Band Cross-Correlation (MF → DCN)\n'
                f'Signals filtered {gamma_band[0]}–{gamma_band[1]} Hz, '
                f'Mean ± SEM across {len(sim_runs)} runs, '
                f'±{max_lag_display*1000:.0f} ms window',
                fontsize=13, fontweight='bold')
ax_gx.legend(loc='upper right', fontsize=10)
ax_gx.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

fig_output_dir = os.path.join('NESTlesions', norm_dir, 'spike synch analysis')
os.makedirs(fig_output_dir, exist_ok=True)
save_figure_multi_format(fig_gamma_xcorr,
                         os.path.join(fig_output_dir, 'gamma_xcorr_MF_DCN'))
print(f"Figure saved to: {fig_output_dir}/gamma_xcorr_MF_DCN.[png|eps|svg]")
plt.show()

# ---- Per-condition subplots ----
fig_gamma_xcorr_sub, axes_gx = plt.subplots(1, n_cond, figsize=(5 * n_cond, 4),
                                             sharey=True)
if n_cond == 1:
    axes_gx = [axes_gx]

for ci, condition in enumerate(conditions):
    ax = axes_gx[ci]
    stack_gx = np.array(gamma_xcorr_mf_dcn[condition])[:, mask_gx]
    mean_gx = stack_gx.mean(axis=0)
    sem_gx = stack_gx.std(axis=0) / np.sqrt(len(stack_gx))

    col = lesion_colors[condition]
    lab = lesion_labels[condition]
    ax.plot(lags_ms_gx, mean_gx, color=col, linewidth=2)
    ax.fill_between(lags_ms_gx, mean_gx - sem_gx, mean_gx + sem_gx,
                    color=col, alpha=0.2)
    ax.axvline(0, color='grey', linestyle='--', linewidth=0.8)
    ax.set_title(lab, fontsize=12, fontweight='bold')
    ax.set_xlabel('Lag (ms)')
    ax.grid(True, linestyle='--', alpha=0.5)

    # Annotate peak lag
    peak_idx = np.argmax(mean_gx)
    peak_lag = lags_ms_gx[peak_idx]
    peak_r = mean_gx[peak_idx]
    ax.axvline(peak_lag, color='darkorange', linestyle=':', linewidth=1.2)
    ax.scatter([peak_lag], [peak_r], color='darkorange', s=40, zorder=6,
               edgecolors='black', linewidths=0.6)
    ax.text(peak_lag + 1, peak_r, f'{peak_lag:+.1f} ms',
            fontsize=9, color='darkorange', fontweight='bold')

axes_gx[0].set_ylabel('Cross-correlation (r)', fontsize=11)

fig_gamma_xcorr_sub.suptitle(
    f'Gamma-Band Cross-Correlation (MF → DCN) per Condition\n'
    f'{gamma_band[0]}–{gamma_band[1]} Hz filtered, '
    f'Mean ± SEM across {len(sim_runs)} runs',
    fontsize=14, fontweight='bold', y=1.06)
plt.tight_layout()

save_figure_multi_format(fig_gamma_xcorr_sub,
                         os.path.join(fig_output_dir, 'gamma_xcorr_MF_DCN_per_condition'))
print(f"Figure saved to: {fig_output_dir}/gamma_xcorr_MF_DCN_per_condition.[png|eps|svg]")
plt.show()

In [ ]:
############################################
# -------- Cross-Spectral Density ----------
# Compute CSD per run (hemisphere-averaged),
# then plot mean ± STD
############################################

import os
from scipy.signal import csd as scipy_csd

nperseg = 1024   # Welch window size

# Compute CSD magnitude for every (condition, run) pair — average across hemispheres
csd_per_run = {cond: [] for cond in conditions}
freqs = None

for condition in conditions:
    for run in sorted(spikes_all[condition].keys()):
        data = spikes_all[condition][run]

        hemi_csds = []
        for hemi in hemispheres:
            mf_rate, _ = population_rate(data[f'MF_{hemi}'], data['duration'], dt)
            dcn_rate, _ = population_rate(data[f'DCN_{hemi}'], data['duration'], dt)

            f, Pxy = scipy_csd(mf_rate, dcn_rate, fs=fs, nperseg=nperseg)
            hemi_csds.append(np.abs(Pxy))

            if freqs is None:
                freqs = f

        csd_per_run[condition].append(np.mean(hemi_csds, axis=0))

# Labels and colours matching the earlier plots
csd_labels = {
    'COSIM':          'CONTROL',
    'COSIM_PKJtoDCN': 'PCtoCN',
    'COSIM_MOStoDCN': 'MOStoCN',
    'COSIM_INHtoPKJ': 'MLItoPC',
    'COSIM_MLItoMLI': 'MLItoMLI',
}
csd_colors = {
    'COSIM':          'darkgray',
    'COSIM_PKJtoDCN': 'lightblue',
    'COSIM_MOStoDCN': 'darkblue',
    'COSIM_INHtoPKJ': 'darkgreen',
    'COSIM_MLItoMLI': 'lightgreen',
}

# Output path for saving the figure in multiple formats
csd_output_dir = os.path.join('NESTlesions', norm_dir, 'spike synch analysis')
os.makedirs(csd_output_dir, exist_ok=True)
csd_output_path = os.path.join(csd_output_dir, 'CSD_MF_DCN_logscale')

plot_cross_spectral_density_mean_std(
    csd_per_run, freqs,
    gamma_band=gamma_band,
    condition_labels=csd_labels,
    condition_colors=csd_colors,
    output_path=csd_output_path,
)

In [ ]:
############################################
# -------- Coherence: MOS-CNe -------------
# Compute per-run coherence (hemisphere-averaged),
# plot mean ± STD + gamma-band barplot
############################################

from scipy.signal import coherence as scipy_coherence

# Compute coherence for every (condition, run) pair — average across hemispheres
coh_per_run = {cond: [] for cond in conditions}
coh_freqs = None

for condition in conditions:
    for run in sorted(spikes_all[condition].keys()):
        data = spikes_all[condition][run]

        hemi_cohs = []
        for hemi in hemispheres:
            mf_rate, _ = population_rate(data[f'MF_{hemi}'], data['duration'], dt)
            dcn_rate, _ = population_rate(data[f'DCN_{hemi}'], data['duration'], dt)

            f_coh, Cxy = scipy_coherence(mf_rate, dcn_rate, fs=fs, nperseg=nperseg)
            hemi_cohs.append(Cxy)

            if coh_freqs is None:
                coh_freqs = f_coh

        coh_per_run[condition].append(np.mean(hemi_cohs, axis=0))

# ---- Prepare data ----
gamma_mask_coh = (coh_freqs >= gamma_band[0]) & (coh_freqs <= gamma_band[1])
freq_mask_coh = coh_freqs <= 100

gamma_means_coh = {}
gamma_stds_coh = {}
gamma_runs_coh = {}
coh_stats = {}   # (mean, std) per condition

for cond, run_cohs in coh_per_run.items():
    stack = np.array(run_cohs)
    mean_coh = stack.mean(axis=0)
    std_coh = stack.std(axis=0)
    coh_stats[cond] = (mean_coh, std_coh)

    run_gamma_avgs = stack[:, gamma_mask_coh].mean(axis=1)
    gamma_runs_coh[cond] = run_gamma_avgs
    gamma_means_coh[cond] = run_gamma_avgs.mean()
    gamma_stds_coh[cond] = run_gamma_avgs.std()

# ---- Figure: Coherence (left) + gamma-bar (right) ----
fig, (ax_coh, ax_bar) = plt.subplots(1, 2, figsize=(14, 6),
                                      gridspec_kw={'width_ratios': [3, 1]})

# -- Left: Coherence mean ± STD --
for cond in conditions:
    mean_coh, std_coh = coh_stats[cond]
    col = csd_colors[cond]
    lab = csd_labels[cond]
    ax_coh.plot(coh_freqs[freq_mask_coh], mean_coh[freq_mask_coh],
                label=lab, color=col, linewidth=2)
    ax_coh.fill_between(coh_freqs[freq_mask_coh],
                        (mean_coh - std_coh)[freq_mask_coh],
                        (mean_coh + std_coh)[freq_mask_coh],
                        color=col, alpha=0.2)

ax_coh.axvspan(gamma_band[0], gamma_band[1], color='gray', alpha=0.15,
               label='Gamma Band')
ax_coh.set_xlim(0, 100)
ax_coh.set_ylim(0, 1)
ax_coh.set_xlabel('Frequency (Hz)')
ax_coh.set_ylabel('Coherence')
ax_coh.set_title('Coherence: MOS\u2013CNe\n(mean ± STD across runs, hemisphere-averaged)')
ax_coh.legend()
ax_coh.grid(True, linestyle='--', alpha=0.6)

# -- Right: barplot of mean gamma-band coherence --
x_pos_coh = np.arange(len(conditions))
bar_means_coh = [gamma_means_coh[c] for c in conditions]
bar_stds_coh = [gamma_stds_coh[c] for c in conditions]
bar_colors_coh = [csd_colors[c] for c in conditions]
bar_labels_coh = [csd_labels[c] for c in conditions]

ax_bar.bar(x_pos_coh, bar_means_coh, yerr=bar_stds_coh, capsize=5, width=0.55,
           color=bar_colors_coh, alpha=0.7, edgecolor='black', linewidth=0.8)

rng = np.random.default_rng(42)
for ci, cond in enumerate(conditions):
    vals = gamma_runs_coh[cond]
    jitter = rng.uniform(-0.12, 0.12, size=len(vals))
    ax_bar.scatter(np.full(len(vals), ci) + jitter, vals,
                   color=darken_color(bar_colors_coh[ci]), s=12, zorder=5, alpha=0.8)

ax_bar.set_xticks(x_pos_coh)
ax_bar.set_xticklabels(bar_labels_coh, rotation=20, ha='right')
ax_bar.set_ylabel('Mean coherence in gamma band')
ax_bar.set_title(f'Gamma-band Coherence\n({gamma_band[0]}–{gamma_band[1]} Hz)')
ax_bar.grid(True, axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()

import os
from NESTlesions.plot_utils import save_figure_multi_format
fig_output_dir = os.path.join('NESTlesions', norm_dir, 'spike synch analysis')
os.makedirs(fig_output_dir, exist_ok=True)
save_figure_multi_format(fig, os.path.join(fig_output_dir, 'coherence_MF_DCN'))
print(f"Figure saved to: {fig_output_dir}/coherence_MF_DCN.[png|eps|svg]")

plt.show()

## Statistical Analysis with FDR Correction and Cohen's d Effect Sizes

Comprehensive statistical comparisons (t-test, Mann-Whitney U) for all spike synchronization metrics across lesion conditions, with:
- **Benjamini-Hochberg FDR correction** applied globally across all comparisons
- **Cohen's d** effect sizes with interpretation
- Results saved to CSV and formatted TXT files

In [ ]:
############################################
# -------- Statistical Analysis with FDR --
# Correction and Cohen's d Effect Sizes ----
############################################

import os
import pandas as pd
from scipy import stats
from scipy.signal import csd as scipy_csd
from statsmodels.stats.multitest import multipletests
from NESTlesions.plot_utils import print_statistical_tests

# Map condition keys to readable labels
condition_labels = [lesion_labels.get(c, c) for c in conditions]

# Collect ALL statistical comparison results
all_statistical_results = []

print("=" * 80)
print("STATISTICAL ANALYSIS WITH FDR CORRECTION AND COHEN'S d")
print("=" * 80)

# ============================================================
# 1. Scalar spike synchronization metrics (from cell 3)
# ============================================================
for m in scalar_metric_names:
    individual_points = [np.array(cond_values[c][m]) for c in conditions]
    results = print_statistical_tests(
        'Spike Synchronization',
        metric_labels[m].replace('\n', ' '),
        condition_labels, individual_points, reference_idx=0
    )
    all_statistical_results.extend(results)

# ============================================================
# 2. Coherence gamma-band (MOS-CNe)
# ============================================================
coh_individual_points = [gamma_runs_coh[c] for c in conditions]
results = print_statistical_tests(
    'MOS-CNe Coherence',
    f'Gamma band ({gamma_band[0]}-{gamma_band[1]} Hz)',
    condition_labels, coh_individual_points, reference_idx=0
)
all_statistical_results.extend(results)

# ============================================================
# 3. CSD gamma-band (MF-DCN) - recompute per-run gamma means
#    (hemisphere-averaged)
# ============================================================
csd_gamma_per_cond = {cond: [] for cond in conditions}
gamma_mask_csd = None

for condition in conditions:
    for run in sorted(spikes_all[condition].keys()):
        data = spikes_all[condition][run]

        hemi_csd_gamma = []
        for hemi in hemispheres:
            mf_rate, _ = population_rate(data[f'MF_{hemi}'], data['duration'], dt)
            dcn_rate, _ = population_rate(data[f'DCN_{hemi}'], data['duration'], dt)

            f_csd, Pxy = scipy_csd(mf_rate, dcn_rate, fs=fs, nperseg=nperseg)
            csd_mag = np.abs(Pxy)

            if gamma_mask_csd is None:
                gamma_mask_csd = (f_csd >= gamma_band[0]) & (f_csd <= gamma_band[1])
            hemi_csd_gamma.append(csd_mag[gamma_mask_csd].mean())

        csd_gamma_per_cond[condition].append(np.mean(hemi_csd_gamma))

csd_individual_points = [np.array(csd_gamma_per_cond[c]) for c in conditions]
results = print_statistical_tests(
    'MOS-CNe Cross-Spectral Density',
    f'Gamma band ({gamma_band[0]}-{gamma_band[1]} Hz)',
    condition_labels, csd_individual_points, reference_idx=0
)
all_statistical_results.extend(results)

# ============================================================
# 4. Amplitude Envelope Correlation (MF-DCN gamma band)
# ============================================================
aec_individual_points = [np.array(aec_values[c]) for c in conditions]
results = print_statistical_tests(
    'Amplitude Envelope Correlation',
    f'MF-DCN Gamma AEC ({gamma_band[0]}-{gamma_band[1]} Hz)',
    condition_labels, aec_individual_points, reference_idx=0
)
all_statistical_results.extend(results)

# ============================================================
# Apply FDR correction (Benjamini-Hochberg) across ALL tests
# ============================================================
stats_df = pd.DataFrame(all_statistical_results)

raw_p_ttest = stats_df['p_value_ttest'].values
raw_p_mannwhitney = stats_df['p_value_mannwhitney'].values

# FDR correction on t-test p-values
_, p_ttest_fdr, _, _ = multipletests(raw_p_ttest, method='fdr_bh')
stats_df['p_value_ttest_fdr'] = p_ttest_fdr

# FDR correction on Mann-Whitney p-values
_, p_mannwhitney_fdr, _, _ = multipletests(raw_p_mannwhitney, method='fdr_bh')
stats_df['p_value_mannwhitney_fdr'] = p_mannwhitney_fdr

# Significance markers for FDR-corrected p-values
stats_df['significance_fdr'] = stats_df['p_value_ttest_fdr'].apply(get_significance_marker)

# Reorder columns (matching lesion_analysis format)
new_order = ['analysis', 'measure', 'condition_1', 'condition_2', 'n_1', 'n_2',
             'mean_1', 'mean_2', 'std_1', 'std_2', 't_statistic',
             'p_value_ttest', 'p_value_ttest_fdr', 'significance', 'significance_fdr',
             'u_statistic', 'p_value_mannwhitney', 'p_value_mannwhitney_fdr',
             'cohens_d', 'effect_size_interpretation']
stats_df = stats_df[new_order]

# ============================================================
# Save results to CSV and formatted TXT
# ============================================================
output_dir = os.path.join('NESTlesions', norm_dir, 'spike synch analysis')
os.makedirs(output_dir, exist_ok=True)

# CSV
csv_path = os.path.join(output_dir, 'statistical_comparisons.csv')
stats_df.to_csv(csv_path, index=False)

# Formatted TXT
txt_path = os.path.join(output_dir, 'statistical_comparisons.txt')
with open(txt_path, 'w') as f:
    f.write("SPIKE SYNCHRONIZATION - STATISTICAL COMPARISONS SUMMARY\n")
    f.write("=" * 80 + "\n\n")
    f.write("Note: FDR correction applied using Benjamini-Hochberg method\n")
    f.write("Note: All metrics are hemisphere-averaged (Left & Right) per run\n")
    f.write(f"Total number of comparisons: {len(stats_df)}\n")
    f.write("=" * 80 + "\n\n")

    for idx, result in stats_df.iterrows():
        f.write(f"Analysis: {result['analysis']}\n")
        f.write(f"Measure: {result['measure']}\n")
        f.write(f"Comparison: {result['condition_1']} vs {result['condition_2']}\n")
        f.write(f"  N (group 1): {result['n_1']}, N (group 2): {result['n_2']}\n")
        f.write(f"  Mean (group 1): {result['mean_1']:.4f} +/- {result['std_1']:.4f}\n")
        f.write(f"  Mean (group 2): {result['mean_2']:.4f} +/- {result['std_2']:.4f}\n")
        f.write(f"  T-test: t={result['t_statistic']:.3f}, p={result['p_value_ttest']:.6f} (raw) [{result['significance']}]\n")
        f.write(f"          p={result['p_value_ttest_fdr']:.6f} (FDR-corrected) [{result['significance_fdr']}]\n")
        f.write(f"  Mann-Whitney U: U={result['u_statistic']:.3f}, p={result['p_value_mannwhitney']:.6f} (raw)\n")
        f.write(f"                  p={result['p_value_mannwhitney_fdr']:.6f} (FDR-corrected)\n")
        f.write(f"  Cohen's d: {result['cohens_d']:.3f} ({result['effect_size_interpretation']} effect)\n")
        f.write("-" * 80 + "\n")

print(f"\n{'='*70}")
print(f"Results saved to:")
print(f"  CSV: {csv_path}")
print(f"  TXT: {txt_path}")
print(f"Total comparisons: {len(stats_df)}")
print(f"Note: Includes raw p-values, FDR-corrected p-values, and Cohen's d")
print(f"{'='*70}")

# Print summary table
print("\n\nSUMMARY TABLE")
print("=" * 140)
header = f"{'Analysis':<32} {'Measure':<35} {'Comparison':<22} {'Cohen d':>8} {'Effect':>12} {'p(raw)':>10} {'p(FDR)':>10} {'Sig':>5}"
print(header)
print("-" * 140)
for _, row in stats_df.iterrows():
    comp = f"{row['condition_1']} vs {row['condition_2']}"
    print(f"{row['analysis']:<32} {row['measure']:<35} {comp:<22} "
          f"{row['cohens_d']:>8.3f} {row['effect_size_interpretation']:>12} "
          f"{row['p_value_ttest']:>10.4f} {row['p_value_ttest_fdr']:>10.4f} {row['significance_fdr']:>5}")
print("=" * 140)

In [ ]:
############################################
# -------- Summary: DCN-DCN PLV + DCN & PKJ Sync -------
# 1×3: DCN–DCN gamma PLV | DCN corr sync | PKJ corr sync
# Stats use FDR-corrected p-values (t-test)
############################################

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from NESTlesions.plot_utils import add_stat_annotation, save_figure_multi_format

# ---------- Helper: barplot with individual points + FDR stats ----------

def _bar_panel_fdr(ax, metric_key, ylabel, title, ylim_bottom=None):
    """
    Barplot of a scalar metric across conditions with individual-run
    data points and FDR-corrected statistical annotations (t-test).
    """
    means = [np.mean(cond_values[c][metric_key]) for c in conditions]
    sds   = [np.std(cond_values[c][metric_key])  for c in conditions]
    x_pos = np.arange(n_cond)

    ax.bar(x_pos, means, yerr=sds, capsize=4, width=0.5,
           color=[lesion_colors[c] for c in conditions], alpha=0.7,
           edgecolor='black', linewidth=0.8)

    rng = np.random.default_rng(42)
    for ci, cond in enumerate(conditions):
        vals = cond_values[cond][metric_key]
        jitter = rng.uniform(-0.1, 0.1, size=len(vals))
        ax.scatter(np.full(len(vals), ci) + jitter, vals,
                   color=darken_color(lesion_colors[cond]), s=10, zorder=5, alpha=0.8)

    ax.set_xticks(x_pos)
    ax.set_xticklabels([lesion_labels.get(c, c) for c in conditions],
                       rotation=25, ha='right', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')

    # FDR-corrected stat annotations (t-test)
    measure_name = metric_labels[metric_key].replace('\n', ' ')
    max_y = max(means[i] + sds[i] for i in range(n_cond))
    spacing = 0.08 * max_y if max_y > 0 else 0.01
    for ti, tc in enumerate(test_conditions):
        ci_test = conditions.index(tc)
        tc_label = lesion_labels[tc]
        mask = ((stats_df['measure'] == measure_name) &
                (stats_df['condition_2'] == tc_label))
        p_fdr = stats_df.loc[mask, 'p_value_ttest_fdr'].values[0] if mask.any() else 1.0
        add_stat_annotation(ax, 0, ci_test, max_y, p_fdr, h=spacing * (ti + 1))

    # Disable ylim_bottom if any data (mean-sd or individual points) falls below it
    if ylim_bottom is not None:
        min_bar = min(means[i] - sds[i] for i in range(n_cond))
        min_pts = min(np.min(cond_values[c][metric_key]) for c in conditions)
        if min(min_bar, min_pts) < ylim_bottom:
            ylim_bottom = None

    ax.set_ylim(bottom=ylim_bottom,
                top=max_y + spacing * (len(test_conditions) + 2))


# ---------- Create 1×3 figure ----------

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(10, 5))

# ---- A: DCN–DCN gamma PLV ----
_bar_panel_fdr(ax_a, 'PLV_DCN_to_DCN', 'PLV', 'CNe–CNe Gamma PLV')

# ---- B: DCN pairwise corr synchrony ----
_bar_panel_fdr(ax_b, 'DCN_internal_sync', 'Pairwise Corr.', 'CNe Pairwise Corr. Synchrony',
               ylim_bottom=0.5)

# ---- C: PKJ pairwise corr synchrony ----
#_bar_panel_fdr(ax_c, 'PC_internal_sync', 'Pairwise Corr.', 'PKJ Pairwise Corr. Synchrony',
#               ylim_bottom=0.5)

# ---- Panel labels ----
for label, ax in zip('AB', [ax_a, ax_b]):
    ax.text(-0.06, 1.10, label, transform=ax.transAxes,
            fontsize=24, fontweight='bold', va='top')

plt.tight_layout()

# ---- Save ----
fig_output_dir = os.path.join('NESTlesions', norm_dir, 'spike synch analysis')
os.makedirs(fig_output_dir, exist_ok=True)
save_figure_multi_format(fig, os.path.join(fig_output_dir, 'summary_DCN_PLV_and_sync'))
print(f"Figure saved to: {fig_output_dir}/summary_DCN_PLV_and_sync.[png|eps|svg]")
plt.show()

In [ ]:
############################################
# -------- Summary: MF–DCN PLV + Gamma Cross-Correlograms + Pairwise Corr Barplot --------
# A (left):         MF–DCN gamma PLV barplot with FDR-corrected t-test
# B (centre):       Gamma-filtered MF→DCN cross-correlograms (all conditions)
# C (right):        MF–DCN pairwise correlation barplot with FDR-corrected t-test
############################################

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from NESTlesions.plot_utils import add_stat_annotation, save_figure_multi_format

fig_summary = plt.figure(figsize=(20, 6))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 2, 1], wspace=0.35)

# ---- Panel A: MF–DCN gamma PLV barplot with FDR stats ----
ax_plv = fig_summary.add_subplot(gs[0])

metric_key = 'PLV_MF_to_DCN'
means = [np.mean(cond_values[c][metric_key]) for c in conditions]
sds   = [np.std(cond_values[c][metric_key])  for c in conditions]
x_pos = np.arange(n_cond)

ax_plv.bar(x_pos, means, yerr=sds, capsize=4, width=0.5,
           color=[lesion_colors[c] for c in conditions], alpha=0.7,
           edgecolor='black', linewidth=0.8)

rng_plv = np.random.default_rng(42)
for ci, cond in enumerate(conditions):
    vals = np.array(cond_values[cond][metric_key])
    jitter = rng_plv.uniform(-0.1, 0.1, size=len(vals))
    ax_plv.scatter(np.full(len(vals), ci) + jitter, vals,
                   color=darken_color(lesion_colors[cond]), s=10, zorder=5, alpha=0.8)

ax_plv.set_xticks(x_pos)
ax_plv.set_xticklabels([lesion_labels.get(c, c) for c in conditions],
                       rotation=25, ha='right', fontsize=13)
ax_plv.set_ylabel('PLV', fontsize=14)
ax_plv.set_title('MOS\u2013CNe Gamma PLV',
                 fontsize=15, fontweight='bold')
ax_plv.tick_params(axis='both', labelsize=13)
ax_plv.grid(True, axis='y', linestyle='--', alpha=0.6)

# FDR-corrected t-test annotations
measure_name = metric_labels[metric_key].replace('\n', ' ')
max_y = max(means[i] + sds[i] for i in range(n_cond))
spacing = 0.08 * max_y if max_y > 0 else 0.01
for ti, tc in enumerate(test_conditions):
    ci_test = conditions.index(tc)
    tc_label = lesion_labels[tc]
    mask_df = ((stats_df['measure'] == measure_name) &
               (stats_df['condition_2'] == tc_label))
    p_fdr = stats_df.loc[mask_df, 'p_value_ttest_fdr'].values[0] if mask_df.any() else 1.0
    add_stat_annotation(ax_plv, 0, ci_test, max_y, p_fdr, h=spacing * (ti + 1))

ax_plv.set_ylim(top=max_y + spacing * (len(test_conditions) + 2))

# ---- Panel B: Gamma-filtered cross-correlograms (MF → DCN) ----
ax_gxcorr = fig_summary.add_subplot(gs[1])

max_lag_display = 0.05  # ±50 ms

mask_gx_s = (gamma_xcorr_lags >= -max_lag_display) & (gamma_xcorr_lags <= max_lag_display)
lags_ms_gx_s = gamma_xcorr_lags[mask_gx_s] * 1000

for condition in conditions:
    stack_gx_s = np.array(gamma_xcorr_mf_dcn[condition])[:, mask_gx_s]
    mean_gx_s = stack_gx_s.mean(axis=0)
    sem_gx_s = stack_gx_s.std(axis=0) / np.sqrt(len(stack_gx_s))

    col = lesion_colors[condition]
    lab = lesion_labels[condition]
    ax_gxcorr.plot(lags_ms_gx_s, mean_gx_s, color=col, linewidth=2, label=lab)
    ax_gxcorr.fill_between(lags_ms_gx_s, mean_gx_s - sem_gx_s, mean_gx_s + sem_gx_s,
                           color=col, alpha=0.15)

ax_gxcorr.axvline(0, color='grey', linestyle='--', linewidth=1)
ax_gxcorr.set_xlabel('Lag (ms)', fontsize=14)
ax_gxcorr.set_ylabel('Cross-correlation (r)', fontsize=14)
ax_gxcorr.set_title(f'Gamma-Filtered ({gamma_band[0]}\u2013{gamma_band[1]} Hz) MOS \u2192 CNe\n'
                    f'Mean \u00b1 SEM across {len(sim_runs)} runs, '
                    f'\u00b1{max_lag_display*1000:.0f} ms window',
                    fontsize=15, fontweight='bold')
ax_gxcorr.legend(loc='upper left', fontsize=13)
ax_gxcorr.tick_params(axis='both', labelsize=13)
ax_gxcorr.grid(True, linestyle='--', alpha=0.5)

# ---- Panel C: MOS–DCN pairwise correlation barplot with FDR stats ----
ax_bar = fig_summary.add_subplot(gs[2])

metric_key = 'MF_DCN_cross_sync'
means = [np.mean(cond_values[c][metric_key]) for c in conditions]
sds   = [np.std(cond_values[c][metric_key])  for c in conditions]
x_pos = np.arange(n_cond)

ax_bar.bar(x_pos, means, yerr=sds, capsize=4, width=0.5,
           color=[lesion_colors[c] for c in conditions], alpha=0.7,
           edgecolor='black', linewidth=0.8)

rng = np.random.default_rng(42)
for ci, cond in enumerate(conditions):
    vals = np.array(cond_values[cond][metric_key])
    jitter = rng.uniform(-0.1, 0.1, size=len(vals))
    ax_bar.scatter(np.full(len(vals), ci) + jitter, vals,
                   color=darken_color(lesion_colors[cond]), s=10, zorder=5, alpha=0.8)

ax_bar.set_xticks(x_pos)
ax_bar.set_xticklabels([lesion_labels.get(c, c) for c in conditions],
                       rotation=25, ha='right', fontsize=13)
ax_bar.set_ylabel('Pairwise Corr.', fontsize=14)
ax_bar.set_title('MOS\u2013CNe\nPairwise Corr. Synchrony',
                 fontsize=15, fontweight='bold')
ax_bar.tick_params(axis='both', labelsize=13)
ax_bar.grid(True, axis='y', linestyle='--', alpha=0.6)

# FDR-corrected t-test annotations (each lesion vs control)
measure_name = metric_labels[metric_key].replace('\n', ' ')
max_y = max(means[i] + sds[i] for i in range(n_cond))
spacing = 0.08 * max_y if max_y > 0 else 0.01
for ti, tc in enumerate(test_conditions):
    ci_test = conditions.index(tc)
    tc_label = lesion_labels[tc]
    mask_df = ((stats_df['measure'] == measure_name) &
               (stats_df['condition_2'] == tc_label))
    p_fdr = stats_df.loc[mask_df, 'p_value_ttest_fdr'].values[0] if mask_df.any() else 1.0
    add_stat_annotation(ax_bar, 0, ci_test, max_y, p_fdr, h=spacing * (ti + 1))

ax_bar.set_ylim(top=max_y + spacing * (len(test_conditions) + 2))

# ---- Panel labels ----
for label, ax in zip('AB', [ax_plv, ax_gxcorr]):
    ax.text(-0.06, 1.10, label, transform=ax.transAxes,
            fontsize=20, fontweight='bold', va='top')
ax_bar.text(-0.25, 1.10, 'C', transform=ax_bar.transAxes,
            fontsize=20, fontweight='bold', va='top')

plt.tight_layout()

# ---- Save ----
fig_output_dir = os.path.join('NESTlesions', norm_dir, 'spike synch analysis')
os.makedirs(fig_output_dir, exist_ok=True)
save_figure_multi_format(fig_summary, os.path.join(fig_output_dir,
                         'summary_PLV_xcorr_and_pairwise_corr'))
print(f"Figure saved to: {fig_output_dir}/summary_PLV_xcorr_and_pairwise_corr.[png|eps|svg]")
plt.show()